In [30]:
# STEP 1: Import the necessary modules.

#mediapipe 0.10.35, most recent one compatable with any python version that's self respecting (since 2023)
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# STEP 2: Create an HandLandmarker object.

# Install hand_landmarker.task from https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/index#models
# Under Models click the link for hand marker, put that file in the same directory as this ipynb notebook
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    
    # equivalent to static_image_mode=True
    running_mode=vision.RunningMode.IMAGE,

    # equivalent to max_num_hands=1
    num_hands=1,

    # equivalent to min_detection_confidence=0.5
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)


In [35]:
import cv2
import numpy as np

# MediaPipe hand connections (same as official spec)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (0, 17),
    (17, 18), (18, 19), (19, 20)
]


def draw_landmarks_on_image(rgb_image, detection_result):
    image = np.copy(rgb_image)
    h, w, _ = image.shape

    for hand_landmarks in detection_result.hand_landmarks:
        # Convert normalized coords -> pixel coords
        points = []
        for lm in hand_landmarks:
            x, y = int(lm.x * w), int(lm.y * h)
            points.append((x, y))

        # Draw points
        for x, y in points:
            cv2.circle(image, (x, y), 5, (0, 255, 0), -1)

        # Draw connections
        for a, b in HAND_CONNECTIONS:
            if a < len(points) and b < len(points):
                cv2.line(image, points[a], points[b], (255, 0, 0), 2)

    return image

In [40]:
from tqdm import tqdm

SEQUENCE_LENGTH = 37
NUM_LANDMARKS = 21
FEATURES_PER_FRAME = NUM_LANDMARKS * 3  # x, y, z = 63

# Function Nghi made

# Image path - > img input
# update mediapipe
# assumes image_array is rgb
def extract_landmarks_from_image(image_array):

    # Image handling

    if image_array is None:
        return np.zeros(FEATURES_PER_FRAME)

    # Mediapipe Stuff
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_array
    )
    
    detection_result = detector.detect(mp_image)

    #drawn_result = draw_landmarks_on_image(image_array, detection_result)

    #plt.imshow(drawn_result)
    #plt.show()

    if detection_result.hand_landmarks:
        hand_landmarks = detection_result.hand_landmarks[0]

        landmarks = []
        for lm in hand_landmarks:
            landmarks.extend([lm.x, lm.y, lm.z])

        return np.array(landmarks)

    else:
        return np.zeros(FEATURES_PER_FRAME)

In [ ]:
#pyautogui switch tab example
import pyautogui

# DO NOT RUN UNLESS YOU WANT YOUR TAB FORCIBILY SWITCHED
# (on windows, on firefox)
with pyautogui.hold('ctrl'):
    pyautogui.press("tab")

In [50]:
# TODO Implement model which takes in (37, 63) numpy array
# outputs index of class
import random
def callModel(landmarkedVideo):
    # convert to torch tensor
    # pass through model
    # argmax for class
    if random.random() < 0.1:
        return 1
    else:
        return 0

In [51]:
# TODO complete this function:
'''
Need a mapping of class indices to the gestures they represent and actions

will probably look like a function mapping indices to pyautogui commands

Somewhere we need to formally document which training data classes associate with which
gestures and then refer to the google docs on which gestures refer to which browser-control actions
'''

def mapIdxToAction(modelPrediction):
    # Other gesture, do nothing
    if modelPrediction == 0:
        pass

    # switch tab macro
    if modelPrediction == 1:
        with pyautogui.hold('ctrl'):
            pyautogui.press("tab")
        


In [53]:
import cv2
from IPython.display import display, clear_output
from PIL import Image
import numpy as np
import io
import time

# Open webcam
cap = cv2.VideoCapture(-0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")



try:
    while True:
        array = []
        for batchIdx in range(37):
            ret, frame = cap.read()
    
            if not ret:
                print("Failed to grab frame")
                break
    
            # Convert BGR -> RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
            # Convert to PIL image
            img = Image.fromarray(frame_rgb)
            np_img = np.array(img)
    
            # landmarks is a np array of length 63
            landmarks = extract_landmarks_from_image(np_img)
            array.append(landmarks)

            # Sample at 40 fps-ish, instead of at rate of however fast my cpu wants to go
            time.sleep(0.025)
        
        # TODO: batch 37 frames togther and output ()
        landmarkedVideo = np.array(array)
        print(landmarkedVideo)

        # TODO: From (37, 63) batch, make classification and take action
        modelClassPrediction = callModel(landmarkedVideo)
        mapIdxToAction(modelClassPrediction)
        break


except KeyboardInterrupt:
    print("Stopped")

finally:
    cap.release()

[[ 8.40788484e-01  9.10576403e-01  4.15925484e-07 ...  7.26627648e-01
   3.43285024e-01 -1.10120997e-01]
 [ 8.16456676e-01  8.53855491e-01  4.92902473e-07 ...  6.43899500e-01
   3.21271479e-01 -1.04126297e-01]
 [ 8.16456676e-01  8.53855491e-01  4.92902473e-07 ...  6.43899500e-01
   3.21271479e-01 -1.04126297e-01]
 ...
 [ 7.19564557e-01  8.41409087e-01  5.31547244e-07 ...  7.34369576e-01
   3.00867260e-01 -1.07403554e-01]
 [ 7.19564557e-01  8.41409087e-01  5.31547244e-07 ...  7.34369576e-01
   3.00867260e-01 -1.07403554e-01]
 [ 6.94131315e-01  8.69471192e-01  6.13547400e-07 ...  6.97066367e-01
   3.26156825e-01 -1.04345582e-01]]
